In [ ]:
configs_dir = 'configs'

model_config_file = f'{configs_dir}/config.json'
tokenizer_config_file = f'{configs_dir}/tokenizer_configs/config.json'

#### infer (serving)

In [ ]:
# start server

import subprocess


env = os.environ.copy()


# model_names = model_names[:1]

for model_name in model_names:
    args = [
        'python', 'infer_serving.py',
        '--engine', 'vllm',
        '--model_name', model_name,
        '--model_config', model_config_file,
        '--tokenizer_config', tokenizer_config_file
    ]

process = subprocess.Popen(
    args,
    env=env,
    # capture_output=True,        # Убрать если нужно не захватывать вывод, а видеть его в реальном времени
    text=True,                    # возвращает строки, а не байты.
    cwd=cwd                       # Если не задано cwd будет равна папке файла ноутубука (то есть откуда запускается subprocess), 
                                  # задать аргумент если нужно изменить
)

print(f'llm server PID: {process.pid}')
PID = process.pid


In [ ]:
# request generate

import requests

url = "http://127.0.0.1:8000/llm_generate"

gen_request = {
    "conv_dataset": f'{run_dir}/conv_dataset.jsonl',
    "output_dir": f'{run_dir}/res',
    "batch_size": 9999,
    "tokenizer_params": {}
}

response = requests.post(url, json=gen_request)

if response.status_code == 200:
    result = response.json()
    print(result)
else:
    print(f"Error: {response.status_code}, {response.text}")

In [ ]:
# request generate (curl)

# curl -X POST "http://127.0.0.1:8000/llm_generate" \
#      -H "Content-Type: application/json" \
#      -d '{
#            "conv_dataset": "/path/to/dataset.jsonl",
#            "output_dir": "/path/to/output",
#            "batch_size": 16,
#            "tokenizer_params": {
#              "padding": true,
#              "truncation": true
#            }
#          }'

In [ ]:
# остановка сервера через эндпоинт

response = requests.get("http://127.0.0.1:8000/shutdown")
print(response.text)


# остановка сервера через переменную процесса
# process.terminate()  # отправляет SIGTERM
# process.wait()       # ждем завершения

# если не получилось убить прошлыми способами 
# !kill -9 {PID}

#### infer (no serving)

In [ ]:
args = [
    'python', 'infer.py',    # 'python', 'infer.py'   # 'python', '-m', 'infer'
    '--engine', 'vllm',
    '--model_name', model_name,
    '--model_config', model_config_file,
    '--tokenizer_config', tokenizer_config_file,
    '--conv_dataset', f'{run_dir}/conv_dataset.jsonl',
    '--output_dir', run_dir,
    '--batch_size', '9999',
]


# с доступом к процессу для возможности завершения через метод а не эндпоинт
process = subprocess.Popen(
    args,
    env=env,
    # capture_output=True,        # Убрать если нужно не захватывать вывод, а видеть его в реальном времени
    text=True,                    # возвращает строки, а не байты.
    cwd=cwd                       # Если не задано cwd будет равна папке файла ноутубука (то есть откуда запускается subprocess), 
                                  # задать аргумент если нужно изменить
)


# print("STDOUT:", result.stdout)
# print("STDERR:", result.stderr)
# print("Return code:", result.returncode)